In [1]:
import pandas as pd
from datetime import datetime
from forex_python.converter import CurrencyRates
from pyxirr import xirr 

In [2]:
date_format = '%Y-%m-%d'
testing_data_1  = {'ticker' : ['AAPL', 'META', 'GOOG', 'BABA', 'META', 'TSLA', 'AAPL', 'GOOG', 'BABA', 'TSLA', 'META', 'MSFT'], 'date' : [datetime.strptime('2018-09-01', date_format), datetime.strptime('2018-10-01', date_format), datetime.strptime('2018-10-01', date_format), datetime.strptime('2018-10-01', date_format), datetime.strptime('2019-03-01', date_format), datetime.strptime('2019-03-01', date_format), datetime.strptime('2020-06-01', date_format), datetime.strptime('2020-06-01', date_format), datetime.strptime('2020-09-01', date_format), datetime.strptime('2020-09-01', date_format), datetime.strptime('2020-12-01', date_format), datetime.strptime('2021-08-07', date_format)], 'uuid' : ['6b3a', '6b3b', '6b3c', '6b3d', '6b3b', '6b3e', '6b3a', '6b3c', '6b3d', '6b3e','6b3f','6b3g'], 'qty' : [100, 100, 100, 30, -100, 50, -100, -100, -30, -50, 35, 60], 'price' : [120, 90, 130, 80, 140, 150, 90, 150, 160, 100, 135, 140],  'type': ['BUY', 'BUY', 'BUY', 'BUY', 'SELL', 'BUY', 'SELL', 'SELL', 'SELL', 'SELL', 'BUY', 'BUY'], 'currency' : ['USD', 'USD','USD','USD','USD','USD','USD','USD','USD','USD','USD', 'USD']}
testing_data_2 = {'ticker' : ['AAPL', 'META', 'AAPL'], 'date' : [datetime.strptime('2018-09-01', date_format), datetime.strptime('2018-12-01', date_format), datetime.strptime('2019-09-01', date_format)], 'uuid' : ['6b3a', '6b3b', '6b3a'], 'qty' : [100, 100, -100], 'price' : [120, 130, 140], 'type': ['BUY', 'BUY', 'SELL'], 'currency' : ['USD', 'USD','USD']}

In [3]:
df1 = pd.DataFrame.from_dict(testing_data_1)
df2 = pd.DataFrame.from_dict(testing_data_2)

In [31]:
for idx, row in df1.iterrows():
    uuid_row = row['uuid']
    orders = df1[df1['uuid'] == uuid_row]

    #check that orders dataframe is larger than 1
    if orders.shape[0] > 1:
        print(f'ticker: {row["ticker"]}, date: {row["date"]} has inverse order')
    else: 
        print(f'ticker: {row["ticker"]}, date: {row["date"]} has NO inverse order')

ticker: AAPL, date: 2018-09-01 00:00:00 has inverse order
ticker: META, date: 2018-10-01 00:00:00 has inverse order
ticker: GOOG, date: 2018-10-01 00:00:00 has inverse order
ticker: BABA, date: 2018-10-01 00:00:00 has inverse order
ticker: META, date: 2019-03-01 00:00:00 has inverse order
ticker: TSLA, date: 2019-03-01 00:00:00 has inverse order
ticker: AAPL, date: 2020-06-01 00:00:00 has inverse order
ticker: GOOG, date: 2020-06-01 00:00:00 has inverse order
ticker: BABA, date: 2020-09-01 00:00:00 has inverse order
ticker: TSLA, date: 2020-09-01 00:00:00 has inverse order
ticker: META, date: 2020-12-01 00:00:00 has NO inverse order
ticker: MSFT, date: 2021-08-07 00:00:00 has NO inverse order


In [30]:
df1

,ticker,date,uuid,qty,price,type,currency
0,AAPL,2018-09-01,6b3a,100,120,BUY,USD
1,META,2018-10-01,6b3b,100,90,BUY,USD
2,GOOG,2018-10-01,6b3c,100,130,BUY,USD
3,BABA,2018-10-01,6b3d,30,80,BUY,USD
4,META,2019-03-01,6b3b,-100,140,SELL,USD
5,TSLA,2019-03-01,6b3e,50,150,BUY,USD
6,AAPL,2020-06-01,6b3a,-100,90,SELL,USD
7,GOOG,2020-06-01,6b3c,-100,150,SELL,USD
8,BABA,2020-09-01,6b3d,-30,160,SELL,USD
9,TSLA,2020-09-01,6b3e,-50,100,SELL,USD


In [70]:
def construct_portfolio(trading_logs, date):
    """
    Constructs portfolio holdings for a given date

    para trading_logs:
        - type: pandas dataframe
        - descn: pandas dataframe with columns ticker, date, uuid, qty, type, currency (potentially also additional columns)

    para date:
        - type: string
        - format: YYYY-MM-DD
    """
    #create portfolio in form of a dictionary: {'YYYY-MM-DD' : ['META', 'AAPL', etc.]}
    portf = {date: []}

    print('trading_log.empty: ', trading_logs.empty)

    if trading_logs.empty == False:
        print('trading logs are not empty')
        #filter dataframe based on date parameter
        trading_logs = trading_logs[trading_logs['date'] <= date]

        #group rows by uuid and sum by qty column: where resulting qty is larger than 0 corresponds to a portfolio position
        holdings =  trading_logs.groupby('uuid', as_index=False).sum()

        # print('groupy by holdings: ', holdings)

        #only keep rows where qty is larger than 0
        holdings = holdings[holdings['qty'] > 0]

        #iterate through dataframe
        for index, row in holdings.iterrows():
            ticker = trading_logs[trading_logs['uuid'] == row['uuid']]
            portf[date].append(ticker['ticker'].values[0])

    return portf

In [68]:
trading_log = pd.DataFrame(columns=['ticker', 'date', 'uuid', 'qty', 'type', 'price', 'currency'])

In [27]:
def compute_mwr(trading_logs, currency = 'USD', last_payment_date=None, currency_min_date = '1999-01-04'):
    """
    Function that computes the Money-weighted rate of return (via the XIRR (Extended Internal Rate of Return)) based on the date given.
    The calculation is performed in a way that it is assumed that all remaining positions would be sold on the given date, based on the current market value
    (XIRR formula: https://anexen.github.io/pyxirr/functions.html#xirr)

    para trading_logs:
        - type: pandas dataframe
        - descn: dataframe with columns ticker, date, uuid, qty, type (BUY/SELL) and currency
    
    para currency:
        - type: string
        - descn: defines the base currency in which MWR is computed; If there are investements in trading log which are not in the based currency, they will be converted to the base currency
    
    para last_payment_date:
        - type: date string in the format: YYYY-MM-DD
        - descn: Corresponds to d_i in the XIRR formula (https://anexen.github.io/pyxirr/functions.html#xirr). defines the date based on which the MWR is computed. Based on this date, market value of stocks that are still long in portfolio will be taken based on this date
                 If date parameter is None, max available date in trading logs will be taken as end date
    
    para currency_min_date:
        - type: string in the format: YYYY-MM-DD
        - descn: latest date for which historical exchange rates are available: https://theforexapi.com/documentation/
    """
    #assign memory for dates and cashflows, which are needed for xirr computation; note that BUY orders correspond to negative cashflows, whereas SELL orders correspond to positive cash flows
    dates = []
    portf_cfs = []

    #create currency rates object to get historical exchange rates
    c = CurrencyRates()

    #check if last_payment_date is None; if it is not None, we will filter trading_logs based on last_payment_date
    if last_payment_date is not None:
        trading_logs = trading_logs[trading_logs['date'] <= last_payment_date]
    
    #iterate through trading_logs and add to dates and cf lists
    for index, row in trading_logs.iterrows():
        #check via uuid if entry in trading log has an existing inverse order (if BUY order has corresponding SELL order and if SELL order has corresponding BUY order)
        uuid_row = row['uuid']
        orders = df1[df1['uuid'] == uuid_row]

        #check that orders dataframe is larger than 1
        if orders.shape[0] > 1:
            #set default exchange rate
            exchange_rate = 1

            #check if transaction currency is not in the base currency
            if row['currency'] != currency:
                #check if date of transaction is smaller than latest historical date for which exchange data is available
                if row['date'] < datetime.strptime(currency_min_date, '%Y-%m-%d'):
                    exchange_rate = c.get_rate(currency, row['currency'], currency_min_date)
                else:
                    exchange_rate = c.get_rate(currency, row['currency'], row['date'])
                
                #compute cash flow
                cf = abs(row['qty']*row['price']*exchange_rate)

                #if buy order we will multiply the cf by -1
                if row['type'] == 'BUY':
                    cf = cf*-1
                
                #add rows to lists
                portf_cfs.append(cf)
                dates.append(row['date'])

    #check if there are any remaining positions in the portfolio
    max_date = trading_logs['date'].max()

    if last_payment_date is not None:
        max_date = last_payment_date

    #check if there are any positions left in the portfolio
    portf = construct_portfolio(trading_logs, max_date)

    #get list of tickers in portf
    tickers = list(portf.values())

    print(type(tickers))

    print(tickers[0][1])

    for ticker in tickers[0]:
        print(f'ticker left in portf: {ticker}')
    
    mwr = xirr(dates, portf_cfs)

    print(f'positions left in portfolio: ', portf)
    print(f'this is mwr: {mwr}')




In [129]:
log_entry = df1[(df1['ticker'] == 'META')].sort_values(by='date', ascending=True)

date1 = log_entry.iloc[0]['date']
date2 = datetime.strptime('2019-12-01', '%Y-%m-%d')

(date2 - date1).days

426

In [127]:
log_entry

,ticker,date,uuid,qty,price,type,currency
1,META,2018-10-01,6b3b,100,90,BUY,USD
10,META,2020-12-01,6b3f,35,135,BUY,USD


In [107]:
compute_mwr(df1)


trading_log.empty:  False
trading logs are not empty
<class 'list'>
MSFT
ticker left in portf: META
ticker left in portf: MSFT
positions left in portfolio:  {Timestamp('2021-08-07 00:00:00'): ['META', 'MSFT']}
this is mwr: -0.27386034604406273


In [86]:
df1['date'].max()

Timestamp('2020-12-01 00:00:00')

In [80]:
df1

,ticker,date,uuid,qty,price,type,currency
0,AAPL,2018-09-01,6b3a,100,120,BUY,USD
1,META,2018-10-01,6b3b,100,90,BUY,USD
2,GOOG,2018-10-01,6b3c,100,130,BUY,USD
3,BABA,2018-10-01,6b3d,30,80,BUY,USD
4,META,2019-03-01,6b3b,-100,140,SELL,USD
5,TSLA,2019-03-01,6b3e,50,150,BUY,USD
6,AAPL,2020-06-01,6b3a,-100,90,SELL,USD
7,GOOG,2020-06-01,6b3c,-100,150,SELL,USD
8,BABA,2020-09-01,6b3d,-30,160,SELL,USD
9,TSLA,2020-09-01,6b3e,-50,100,SELL,USD


In [14]:
ticker = 'AAPL'
df1[(df1['ticker'] == ticker) & (df1['type'] == 'BUY')].sort_values('date').iloc[0]['uuid']

'6b3a'

In [77]:
df1.to_csv('trading_log_test.csv', index=False)

In [72]:
portf = construct_portfolio(trading_log, '2021-09-01')

trading_log.empty:  True


In [73]:
portf.values()

dict_values([[]])

In [53]:
portf = construct_portfolio(df1, '2022-06-01')
portf

groupy by holdings:     uuid  qty
0  6b3a    0
1  6b3b    0
2  6b3c    0
3  6b3d    0
4  6b3e    0
5  6b3f   35


{'2022-06-01': ['META']}